# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [1]:
# WHAT THIS CELL DOES:
# Bring in the libraries this notebook needs.
# rich prints coloured, formatted text. dotenv reads the .env file. anthropic talks to Claude.
# The last line loads your keys from .env.

# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
import anthropic
import json
load_dotenv(override=True)

True

In [2]:
# WHAT THIS CELL DOES:
# A small helper to print text nicely.
# It tries the rich Console first (colours, strikethrough). If that fails, it uses plain print.

def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
# WHAT THIS CELL DOES:
# Create the connection to Claude. It reads ANTHROPIC_API_KEY from .env.

claude = anthropic.Anthropic()

In [4]:
# WHAT THIS CELL DOES:
# Two empty lists that work as a pair.
# checklist holds the task texts. completed holds True or False for each task, in the same order.

# Some lists!

checklist = []
completed = []

In [5]:
# WHAT THIS CELL DOES:
# Build a text report of the checklist and show it.
# enumerate gives the position (index) and the item together.
# Finished tasks are shown green with a line through them. The report is also returned so Claude can read it.

def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [6]:
# WHAT THIS CELL DOES:
# Try it out. The lists are still empty, so the report is empty.

get_checklist_report()

''

In [9]:
# WHAT THIS CELL DOES:
# Tool 1: add new tasks to the checklist.
# extend adds all the new tasks to the list. For each new task, a False is added to completed.
# It returns the report, so Claude sees the full list.

def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [10]:
# WHAT THIS CELL DOES:
# Tool 2: mark one task as done.
# Claude counts tasks from 1, but Python lists count from 0. That is why we use index - 1.
# If the number is not in the list, it returns an error text instead. It also prints Claude's notes.

def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [12]:
# WHAT THIS CELL DOES:
# Try tool 1 by hand: empty both lists, then add three tasks.

checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [13]:
# WHAT THIS CELL DOES:
# Try tool 2 by hand: mark task 1 as done. It should now show green and crossed out.

mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [14]:
# WHAT THIS CELL DOES:
# Describe tool 1 to Claude: its name, what it does, and its input (a list of task texts).
# Claude cannot see the Python function, so this description is all it knows.

create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "input_schema": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [15]:
# WHAT THIS CELL DOES:
# Describe tool 2 to Claude: its name, what it does, and its two inputs
# (the task number, and notes about how the task was completed).

mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "input_schema": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [16]:
# WHAT THIS CELL DOES:
# Put both tool descriptions in one list. We hand this list to Claude with every request.

tools = [create_checklist_json, mark_complete_json]

In [17]:
# WHAT THIS CELL DOES:
# Run the tools Claude asked for.
# For each request: read the tool name and inputs, find the Python function by name with globals(),
# call it, and pack the result in the format Claude expects back (a "tool_result").

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.name
        arguments = tool_call.input
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"type": "tool_result", "tool_use_id": tool_call.id, "content": json.dumps(result)})
    return results

In [18]:
# WHAT THIS CELL DOES:
# The agent loop. This is the heart of the notebook.
# 1. Send the messages, the system message and the tools to Claude.
# 2. While Claude asks for tools: run them, add the results to the messages, and ask Claude again.
# 3. When Claude stops asking for tools, show its final text answer.
# Each trip around the loop is one step of Claude working through its checklist.

def loop(messages):
    response = claude.messages.create(model="claude-opus-5-5", max_tokens=4096, system=system_message, messages=messages, tools=tools)
    while response.stop_reason == "tool_use":
        tool_calls = [block for block in response.content if block.type == "tool_use"]
        results = handle_tool_calls(tool_calls)
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": results})
        response = claude.messages.create(model="claude-opus-5-5", max_tokens=4096, system=system_message, messages=messages, tools=tools)
    show("".join(block.text for block in response.content if block.type == "text"))

In [19]:
# WHAT THIS CELL DOES:
# Write the instructions and the question.
# system_message tells Claude how to work: make a plan with the checklist, do each step, then answer.
# user_message is the problem to solve. messages holds the conversation, starting with that problem.

system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "user", "content": user_message}]

In [20]:
# WHAT THIS CELL DOES:
# Run it! Empty the checklist first, then start the loop.
# Watch Claude create its checklist and cross off the steps one by one.

checklist, completed = [], []
loop(messages)

Checklist #1: Estimate the Boston–New York train distance (not given)
Checklist #2: Compute the head start of the Boston train before 3:00 pm
Checklist #3: Compute the closing speed and the time to meet after 3:00 pm
Checklist #4: Find the meeting clock time and location, and sanity-check

The Amtrak Northeast Corridor route from Boston South Station to NY Penn Station is about 230 miles, so I'm 
assuming D ≈ 230 mi.

Checklist #1: Estimate the Boston–New York train distance (not given)
Checklist #2: Compute the head start of the Boston train before 3:00 pm
Checklist #3: Compute the closing speed and the time to meet after 3:00 pm
Checklist #4: Find the meeting clock time and location, and sanity-check

From 2:00 to 3:00 pm the Boston train covers 60 mph × 1 h = 60 mi. That leaves 230 − 60 = 170 mi between the trains
at 3:00 pm.

Checklist #1: Estimate the Boston–New York train distance (not given)
Checklist #2: Compute the head start of the Boston train before 3:00 pm
Checklist #3: Compute the closing speed and the time to meet after 3:00 pm
Checklist #4: Find the meeting clock time and location, and sanity-check

The closing speed is 60 + 80 = 140 mph. Time = 170 ÷ 140 ≈ 1.214 h ≈ 1 h 13 min.

Checklist #1: Estimate the Boston–New York train distance (not given)
Checklist #2: Compute the head start of the Boston train before 3:00 pm
Checklist #3: Compute the closing speed and the time to meet after 3:00 pm
Checklist #4: Find the meeting clock time and location, and sanity-check

3:00 pm + 1 h 13 min ≈ 4:13 pm. The Boston train has gone 60 + 72.9 ≈ 132.9 mi and the NY train has gone ≈ 97.1 mi.
These add up to 230 mi, which checks out. The meeting point is on the Connecticut shoreline, between New Haven and 
Old Saybrook.

Checklist #1: Estimate the Boston–New York train distance (not given)
Checklist #2: Compute the head start of the Boston train before 3:00 pm
Checklist #3: Compute the closing speed and the time to meet after 3:00 pm
Checklist #4: Find the meeting clock time and location, and sanity-check

The trains meet at about 4:13 pm, somewhere on the Connecticut shoreline. This assumes the distance between the 
cities is about 230 miles, since the question didn't give it.

How I got there:
1. Distance: The Amtrak rail route from Boston to New York is about 230 miles, so I used that.
2. Boston train's head start: From 2:00 to 3:00 pm it travels 60 miles. That leaves 230 − 60 = 170 miles between 
the trains at 3:00 pm.
3. Speed toward each other: Together they close the gap at 60 + 80 = 140 mph.
4. Time to meet: 170 ÷ 140 ≈ 1.21 hours, or about 1 hour 13 minutes after 3:00 pm.

Where they meet: The Boston train has gone about 133 miles and the New York train about 97 miles. Those add up to 
230 miles, which confirms the answer. That puts the meeting point between New Haven and Old Saybrook, Connecticut.

If you use a different distance, the meeting time is 3:00 pm + (D − 60) ÷ 140 hours. For example, with the 215-mile
driving distance they'd meet at about 4:06 pm.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>